# Notebook 06: Production Workflows & Jobs

**Exam Coverage**: Section 4 (Production Pipelines)

**Duration**: 45-60 minutes

---
## Learning Objectives
By the end of this notebook, you will be able to:
- Create and configure Databricks Jobs through the UI
- Implement multi-task workflows with dependencies
- Configure job parameters using widgets
- Implement error handling with try/except and dbutils.notebook.exit
- Monitor and troubleshoot job execution
- Understand Free Edition job limitations
---

## Section 1: Introduction to Databricks Jobs
**Databricks Jobs** provide orchestration for notebooks, Python scripts, and JARs.
### 🎯 Key Concepts
**Job Components**
| Component | Purpose | Example |
|-----------|---------|--------|
| **Task** | Unit of work | Notebook, Python script, JAR |
| **Dependency** | Execution order | Task B runs after Task A |
| **Cluster** | Compute resources | Job cluster, all-purpose cluster |
| **Schedule** | When to run | Manual, cron, triggered |
| **Parameters** | Runtime config | Environment, date, mode |
### Job Types
1. **Single-Task Job**: One notebook/script
2. **Multi-Task Job**: Multiple tasks with dependencies (DAG)
3. **Triggered Job**: Runs on external events
4. **Scheduled Job**: Cron-based scheduling
### ⚠️ Free Edition Limitations
**Important restrictions:**
- ❌ Maximum **5 tasks** per workflow
- ❌ No job clusters (must use all-purpose clusters)
- ❌ No advanced scheduling
- ❌ Limited concurrency (single-node)

**For production**: Full Databricks workspace recommended.

## Section 2: Job Parameters with Widgets
**Widgets** enable parameterized notebooks that accept runtime configuration.
### Widget Types
| Type | Purpose | Example |
|------|---------|--------|
| `text` | String input | Environment name, date |
| `dropdown` | Select from options | mode: full/incremental |
| `combobox` | Dropdown + free text | Region selection |
| `multiselect` | Multiple selections | Tables to process |
### Widget Syntax
```python
# Create widget
dbutils.widgets.text("name", "default_value", "Label")
# Get widget value
value = dbutils.widgets.get("name")
# Remove widget
dbutils.widgets.remove("name")
# Remove all
dbutils.widgets.removeAll()
```
**Key points:**
- Widgets appear at top of notebook
- Jobs pass parameters via widget names
- Default values used if not provided

In [0]:
# Import shared variables
%run ./variables

Warning you are using the ipython `%run` line magic. To use the databricks `%run` cell magic make sure that the magic is at the very start of the cell.

---
### 🎯 EXERCISE 1: Create Job Parameter Widgets
**Your task**: Create widgets for job parameterization.

**Requirements:**
Create 4 widgets:
1. **environment** (text)
- Default: "dev"
- Label: "Environment (dev/staging/prod)"
2. **pipeline_mode** (dropdown)
- Options: ["full", "incremental"]
- Default: "incremental"
- Label: "Pipeline Mode"
3. **date** (text)
- Default: "" (empty)
- Label: "Processing Date (YYYY-MM-DD, blank=today)"
4. **enable_quality_checks** (dropdown)
- Options: ["true", "false"]
- Default: "true"
- Label: "Enable Quality Checks"

**Functions:**
```python
dbutils.widgets.text("name", "default", "label")
dbutils.widgets.dropdown("name", "default", ["opt1", "opt2"], "label")
```

In [0]:
# TODO: Create job parameter widgets

# 1. Environment widget (text)
dbutils.widgets.text("environment", "dev", "Environment (dev/stagin/prod)")

# 2. Pipeline mode widget (dropdown)
dbutils.widgets.dropdown("pipeline_mode", "incremental", ["full", "incremental"], "Pipeline Mode")

# 3. Date widget (text)
dbutils.widgets.text("date", "", "Processing Date (YYYY-MM-DD, blank=today)")

# 4. Quality checks widget (dropdown)
dbutils.widgets.dropdown("enable_quality_checks", "true", ["true", "false"], "Enable Quality Checks")

print("✓ Widgets created")

✓ Widgets created


---
**Solution below** ⬇️

In [0]:
# ✅ SOLUTION: Create Widgets

# dbutils.widgets.text("environment", "dev", "Environment (dev/staging/prod)")
# dbutils.widgets.dropdown("pipeline_mode", "incremental", ["full", "incremental"], "Pipeline Mode")
# dbutils.widgets.text("date", "", "Processing Date (YYYY-MM-DD, blank=today)")
# dbutils.widgets.dropdown("enable_quality_checks", "true", ["true", "false"], "Enable Quality Checks")

# print("✅ Widgets created")

---
### Reading Widget Values
Now let's read and use the widget values.

In [0]:
# Read widget values
environment = dbutils.widgets.get("environment")
pipeline_mode = dbutils.widgets.get("pipeline_mode")
processing_date = dbutils.widgets.get("date")
enable_quality_checks = dbutils.widgets.get("enable_quality_checks") == "true"

# Use today's date if not specified
if not processing_date:
    from datetime import datetime
    processing_date = datetime.now().strftime("%Y-%m-%d")

print(f"Environment: {environment}")
print(f"Pipeline Mode: {pipeline_mode}")
print(f"Processing Date: {processing_date}")
print(f"Quality Checks: {enable_quality_checks}")

Environment: dev
Pipeline Mode: incremental
Processing Date: 2026-06-01
Quality Checks: True


## Section 3: Task Execution Pattern
Each task in a multi-task job is typically a separate notebook.
### Common Task Pattern
```python
try:
# Task logic
result = execute_task()
# Success
dbutils.notebook.exit("SUCCESS")
except Exception as e:
# Failure
error_msg = str(e)
dbutils.notebook.exit(f"FAILED: {error_msg}")
```
### Task 1: Data Validation
Validates source data availability before processing.

In [0]:
# Task 1: Data Validation - Provided as example

print("=== Task 1: Data Validation ===")

try:
    # Check source data exists
    customers_exists = len(dbutils.fs.ls(CUSTOMERS_LANDING_PATH)) > 0
    products_exists = len(dbutils.fs.ls(PRODUCTS_LANDING_PATH)) > 0
    sales_exists = len(dbutils.fs.ls(SALES_LANDING_PATH)) > 0
    
    print(f"✓ Customers data: {'Found' if customers_exists else 'Missing'}")
    print(f"✓ Products data: {'Found' if products_exists else 'Missing'}")
    print(f"✓ Sales data: {'Found' if sales_exists else 'Missing'}")
    
    if not (customers_exists and products_exists and sales_exists):
        raise Exception("Missing required source data")
    
    # Check Bronze tables exist
    tables_to_check = [
        CUSTOMERS_BRONZE_TABLE,
        PRODUCTS_BRONZE_TABLE,
        SALES_BRONZE_TABLE
    ]
    
    for table in tables_to_check:
        table_exists = spark.catalog.tableExists(table)
        print(f"✓ Table {table}: {'Exists' if table_exists else 'Missing'}")
    
    print("\n✅ Data validation passed")
    validation_result = "SUCCESS"
    
except Exception as e:
    print(f"\n✗ Data validation failed: {str(e)}")
    validation_result = f"FAILED: {str(e)}"

print(f"\nValidation Result: {validation_result}")

=== Task 1: Data Validation ===
✓ Customers data: Found
✓ Products data: Found
✓ Sales data: Found
✓ Table cert_prep_catalog.01_bronze.customers_raw: Exists
✓ Table cert_prep_catalog.01_bronze.products_raw: Exists
✓ Table cert_prep_catalog.01_bronze.sales_raw: Exists

✅ Data validation passed

Validation Result: SUCCESS


## Section 4: Error Handling Patterns
Production jobs need robust error handling.
### Key Patterns
1. **Try-Except Blocks**: Catch and handle errors
2. **dbutils.notebook.exit()**: Pass results to orchestrator
3. **Retry Logic**: Automatic retry for transient failures
4. **Alerting**: Notify on failures
### dbutils.notebook.exit() Usage
```python
# Success
dbutils.notebook.exit("SUCCESS")
# Failure with details
dbutils.notebook.exit(json.dumps({
"status": "FAILED",
"error": "Connection timeout",
"records_processed": 1000
}))
```
**Important**: Return value accessible by downstream tasks.

---
### 🎯 EXERCISE 2: Implement Retry Logic
**Your task**: Create a function that retries failed tasks.
**Requirements:**
Function signature:
```python
def execute_task_with_retry(task_name, task_function, max_retries=3)
```
**Logic:**
1. Loop up to `max_retries` attempts
2. Try to execute `task_function()`
3. On success:
- Return dict with: `{"status": "SUCCESS", "task": task_name, "attempt": N, "timestamp": ..., "result": ...}`
4. On exception:
- If retries remaining: print error, retry
- If max retries reached: return `{"status": "FAILED", "task": task_name, "error": ...}`

**Hint**: Use a while loop and track attempt count.

In [0]:
# TODO: Implement retry logic function

import json
from datetime import datetime

def execute_task_with_retry(task_name, task_function, max_retries=3):
    """
    Execute a task with automatic retry logic.
    
    Args:
        task_name: Name of the task
        task_function: Function to execute
        max_retries: Maximum retry attempts
    
    Returns:
        dict: Task execution result
    """
    attempt = 0
    
    while attempt < max_retries:
        try:
            # TODO: Print attempt info
            print(f"[{task_name}] Attempt {attempt + 1} of {max_retries}")
            # TODO: Execute task function
            result = task_function()
            # TODO: Return success dict
            return({"status": "SUCCESS",
                    "task": task_name,
                    "attempt": attempt + 1, 
                    "timestamp": datetime.now().isoformat(), 
                    "result": result})
            
            
            
            
        except Exception as e:
            # TODO: Increment attempt
            attempt = attempt + 1
            # TODO: Print error
            error_msg = str(e)
            print(f"[{task_name}] Error on attempt {attempt}: {error_msg}")
            # TODO: Check if max retries reached
            if attempt >= max_retries:
                # TODO: Return failure dict
                return({"status": "FAILED",
                        "task": task_name,
                        "attempt": attempt,
                        "timestamp": datetime.now().isoformat(),
                        "error": error_msg})
                
                
                
                
            # TODO: Print retry message
            print(f" {task_name} Retrying task...")

# Test the function
def sample_task():
    print("  Executing task logic...")
    return {"records_processed": 1000}

result = execute_task_with_retry("sample_task", sample_task, max_retries=3)
print(f"\nTask Result:")
print(json.dumps(result, indent=2))

[sample_task] Attempt 1 of 3
  Executing task logic...

Task Result:
{
  "status": "SUCCESS",
  "task": "sample_task",
  "attempt": 1,
  "timestamp": "2026-06-01T19:25:56.000621",
  "result": {
    "records_processed": 1000
  }
}


---
**Solution below** ⬇️

In [0]:
# ✅ SOLUTION: Retry Logic

# import json
# from datetime import datetime

# def execute_task_with_retry(task_name, task_function, max_retries=3):
#     """
#     Execute a task with automatic retry logic.
#     """
#     attempt = 0
    
#     while attempt < max_retries:
#         try:
#             print(f"[{task_name}] Attempt {attempt + 1}/{max_retries}")
            
#             result = task_function()
            
#             return {
#                 "status": "SUCCESS",
#                 "task": task_name,
#                 "attempt": attempt + 1,
#                 "timestamp": datetime.now().isoformat(),
#                 "result": result
#             }
            
#         except Exception as e:
#             attempt += 1
#             error_msg = str(e)
#             print(f"[{task_name}] Error on attempt {attempt}: {error_msg}")
            
#             if attempt >= max_retries:
#                 return {
#                     "status": "FAILED",
#                     "task": task_name,
#                     "attempt": attempt,
#                     "timestamp": datetime.now().isoformat(),
#                     "error": error_msg
#                 }
            
#             print(f"[{task_name}] Retrying...")

# # Test
# def sample_task():
#     print("  Executing task logic...")
#     return {"records_processed": 1000}

# result = execute_task_with_retry("sample_task", sample_task, max_retries=3)
# print(f"\n✅ Task Result:")
# print(json.dumps(result, indent=2))

## Section 5: Creating Multi-Task Jobs in the UI
### 🚀 Step-by-Step Job Creation
**1. Navigate to Workflows**
- Click **Jobs & Pipelines ** in left sidebar
- Click **Job** in the top right corner

**2. Configure Job Settings**
| Setting | Value |
|---------|-------|
| **Job Name** | `cert_prep_pipeline` |
| **Description** | Medallion architecture ETL pipeline |

**3. Add Tasks (Up to 5 for Free Edition)**

**Task 1: Bronze Ingestion**
- Task Name: `bronze_ingestion`
- Type: Notebook
- **Source**: `02_Auto_Loader_Incremental_Ingestion`
- Cluster: Serverless
- Parameters: `pipeline_mode: incremental`

**Task 2: Silver Transformation**
- Task Name: `silver_transformation`
- Type: Notebook
- **Source**: `03_Bronze_to_Silver`
- **Depends On**: `bronze_ingestion`
- Cluster: Serverless
- Parameters: `enable_quality_checks: true`

**Task 3: Gold Aggregation**
- Task Name: `gold_aggregation`
- Type: Notebook
- **Source**: `04_Silver_to_Gold_Advanced`
- **Depends On**: `silver_transformation`
- Cluster: Serverless

**4. Configure Schedule (Optional)**
- **Trigger Type**: Cron
- **Cron Expression**: `0 2 * * *` (daily at 2 AM)
- **Timezone**: Your timezone

Don't forget to stop the job later on if you decide to schedule it

**5. Configure Alerts (Optional)**
- On Failure: Email notification
- On Success: Optional

**6. Save and Run**
- Click **Create**
- Click **Run now** to test
### Task Dependencies Visualization
```
validate_data
↓
bronze_ingestion
↓
silver_transformation
↓
gold_aggregation
↓
summary_report
```
### ⚠️ Free Edition Constraints
- ✅ Maximum **5 tasks** (we use exactly 5)
- ❌ No job clusters (use all-purpose)
- ❌ No email alerts (basic only)
- ❌ Simple cron scheduling only

## Section 6: Monitoring Jobs
The Databricks Jobs UI provides comprehensive monitoring.
### 📊 Job Run Details
For each run:
| View | Information |
|------|-------------|
| **Run Status** | Success, Failed, Running, Cancelled |
| **Task Timeline** | Visual execution timeline |
| **Task DAG** | Dependency graph |
| **Logs** | stdout, stderr per task |
| **Spark UI** | Detailed execution metrics |
| **Parameters** | Runtime parameters used |
### Accessing Job Context Programmatically

In [0]:
# Example: Get current job context (only in job execution)

try:
    job_id = dbutils.notebook.entry_point.getDbutils().notebook().getContext().jobId().get()
    run_id = dbutils.notebook.entry_point.getDbutils().notebook().getContext().currentRunId().get()
    
    print(f"Job ID: {job_id}")
    print(f"Run ID: {run_id}")
except:
    print("ℹ️  Not running in job context (expected in interactive mode)")

## Section 7: Troubleshooting Common Issues
### Issue 1: Task Failure
**Symptoms**: Task shows as failed

**Diagnosis:**
1. Check task logs for error messages
2. Review Spark UI for details
3. Verify cluster resources


**Solutions:**
- Add try-except error handling
- Implement retry logic
- Increase cluster size
### Issue 2: Task Timeout
**Symptoms**: Task runs indefinitely

**Diagnosis:**
1. Check for never-terminating streams
2. Look for blocking operations
3. Review for cartesian products

**Solutions:**
- Add timeout configurations
- Use `trigger(availableNow=True)`
- Optimize queries with filters
### Issue 3: Parameter Passing
**Symptoms**: Widget values not received

**Diagnosis:**
1. Verify widget names match parameters
2. Check for typos

**Solutions:**
- Use consistent naming
- Add default values
- Log parameters at task start
### Issue 4: Dependency Failures
**Symptoms**: Downstream tasks fail

**Diagnosis:**
1. Check task dependencies
2. Review upstream exit codes

**Solutions:**
- Configure conditional dependencies
- Add data validation
- Use "If" conditions on dependencies

## Section 8: Job Best Practices
### 1️⃣ Idempotency
Jobs should produce same result if run multiple times.
- ✅ Use `INSERT OVERWRITE` for full refreshes
- ✅ Use `MERGE` for incremental updates
- ✅ Check for existing data
### 2️⃣ Parameterization
Make jobs configurable.
- ✅ Use widgets for runtime config
- ✅ Avoid hardcoded values
- ✅ Support multiple environments
### 3️⃣ Error Handling
Implement comprehensive error handling.
- ✅ Try-except blocks
- ✅ Retry logic for transient failures
- ✅ Log errors with context
- ✅ Meaningful status codes
### 4️⃣ Monitoring
Set up monitoring for production.
- ✅ Configure failure alerts
- ✅ Track SLA metrics
- ✅ Monitor data quality
- ✅ Job health dashboards
### 5️⃣ Resource Management
Optimize cluster usage.
- ✅ Use job clusters (when available)
- ✅ Right-size for workload
- ✅ Enable autoscaling
- ✅ Use spot instances (non-critical)
### 6️⃣ Testing
Test before production.
- ✅ Test with sample data
- ✅ Verify task dependencies
- ✅ Test failure scenarios
- ✅ Validate parameters

## Section 9: Summary and Checkpoint
### 🎯 Key Concepts Covered

**1. Databricks Jobs**
- Multi-task workflows
- Task dependencies (DAG)
- Job vs all-purpose clusters

**2. Job Parameters**
- Widget creation and usage
- Parameter passing between tasks
- Environment-specific config

**3. Error Handling**
- Try-except patterns
- Retry logic
- `dbutils.notebook.exit()` for task status

**4. Job Configuration**
- UI-based creation
- Task dependencies
- Scheduling and triggers

**5. Monitoring**
- Run details and logs
- Common failure patterns
- Debugging techniques

**6. Free Edition Limits**
- 5 task maximum
- No job clusters
- Limited scheduling

### ✅ Exam Checklist
Can you:
- [ ] Create widgets and read parameter values?
- [ ] Configure multi-task jobs with dependencies?
- [ ] Implement error handling with try-except?
- [ ] Use `dbutils.notebook.exit()` for task status?
- [ ] Explain job cluster vs all-purpose cluster?
- [ ] Configure job scheduling with cron?
- [ ] Monitor and troubleshoot job runs?
### 📚 Next Steps
**To Create Your Job:**
1. ✅ Organize notebooks by task
2. ✅ Navigate to Workflows → Jobs
3. ✅ Create job with 5 tasks
4. ✅ Configure dependencies
5. ✅ Add parameters and schedule
6. ✅ Run and monitor

**🎉 Notebook Complete!**
You've learned production job orchestration. Create your multi-task workflow in the UI to see it in action!

In [0]:
# Clean up widgets
dbutils.widgets.removeAll()
print("✅ Widgets removed")